# Prompt Engineering Lab

## Setup

In [3]:
import os
import json
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

MODEL = "gemini-3.5-flash-lite"
THINKING = types.ThinkingConfig(thinking_level=types.ThinkingLevel.MINIMAL)


## Task 1: Summarization

### Iteration 1 — Baseline

In [4]:
text_to_summarize = (
    "The city council voted 6-3 on Tuesday to approve a $42 million budget for repaving "
    "roughly 38 miles of residential streets over the next two years. Supporters said the "
    "plan targets neighborhoods that have not seen major roadwork in over a decade and "
    "will reduce vehicle repair costs and improve emergency-vehicle access. Opponents "
    "argued the funding should instead go toward public transit expansion, noting that "
    "only 12% of the city's residents drive to work daily."
)

response = client.models.generate_content(
    model=MODEL,
    contents="Summarize this: " + text_to_summarize,
    config=types.GenerateContentConfig(
        thinking_config=THINKING,
        temperature=1.0,
        top_p=0.95,
    ),
)
print(response.text)


**Summary:** 

The city council voted 6-3 to approve a $42 million, two-year budget to repave about 38 miles of neglected residential streets. Supporters argued the project will improve emergency access and reduce vehicle repair costs, while opponents contended the money should be redirected to public transit since only 12% of residents drive to work.


### Iteration 2 — Add context, constraints, lower temperature

In [5]:
refined_prompt = (
    "You are writing for a local newspaper's 60-second briefing section. "
    "Summarize the following article in exactly 2 sentences, in a neutral tone. "
    "Do not add opinions or information not in the source text.\n\n"
    + text_to_summarize
)

response = client.models.generate_content(
    model=MODEL,
    contents=refined_prompt,
    config=types.GenerateContentConfig(
        thinking_config=THINKING,
        system_instruction="You are an expert news editor.",
        temperature=0.2,
        top_p=0.8,
    ),
)
print(response.text)


The city council voted 6-3 on Tuesday to approve a $42 million budget to repave about 38 miles of residential streets over the next two years. While supporters argued the plan will improve emergency access and reduce vehicle repair costs in neglected neighborhoods, opponents contended the funds should be redirected toward public transit expansion.


## Task 2: Email Generation

### Iteration 1 — Baseline

In [6]:
customer_email = (
    "Subject: Order #48213 arrived damaged\n\n"
    "Hi, I ordered a ceramic vase (Order #48213) last week and it arrived today with a "
    "large crack down one side. I paid for expedited shipping because this was meant to "
    "be a birthday gift for this weekend. I would like a replacement shipped overnight, "
    "or a full refund if that is not possible. Please let me know quickly.\n\n- Jordan"
)

response = client.models.generate_content(
    model=MODEL,
    contents="Write a reply to this email: " + customer_email,
    config=types.GenerateContentConfig(
        thinking_config=THINKING,
        temperature=1.0,
        top_p=0.95,
    ),
)
print(response.text)


Here are three options you can use, depending on your company's policy and how you want to handle the situation:

### Option 1: Full service (Sends replacement overnight + refunds shipping) — *Recommended*
**Subject:** Re: Order #48213 arrived damaged

Hi Jordan,

I am so sorry to hear that your vase arrived damaged, especially since it was meant for a birthday gift this weekend! 

We want to make this right immediately. I have just processed a replacement order to be shipped via overnight delivery at no additional cost, and I’ve also refunded your original expedited shipping fee. You should receive tracking information shortly, and the new vase will arrive in time for the weekend. 

Regarding the damaged vase, please feel free to discard it—there is no need to hassle with sending it back. 

If you need anything else at all, please let me know. Happy birthday to your friend!

Best,

[Your Name]  
[Company Name]

***

### Option 2: Photo required first (If your policy requires proof bef

### Iteration 2 — Add role, policy context, constraints

In [7]:
company_policy = (
    "For damaged items reported within 14 days, offer either (a) a free overnight "
    "replacement at no extra cost, or (b) a full refund including original shipping. "
    "Apologize once, be concise, and always restate both options clearly."
)

refined_prompt = f"""Using the policy below, write a reply to the customer email below.

Policy:
{company_policy}

Customer email:
{customer_email}

Constraints:
- Apologize exactly once.
- Clearly restate BOTH options as separate bullet points.
- Keep the email under 120 words.
- Sign off as "The Support Team".
"""

response = client.models.generate_content(
    model=MODEL,
    contents=refined_prompt,
    config=types.GenerateContentConfig(
        thinking_config=THINKING,
        system_instruction="You are a customer support agent for an online home-goods store.",
        temperature=0.4,
        top_p=0.9,
    ),
)
print(response.text)


Subject: Re: Order #48213 arrived damaged

Hi Jordan,

We sincerely apologize that your ceramic vase arrived damaged. Since you reported this within 14 days, we can resolve this for you right away. 

Please choose one of the following options:
* A free overnight replacement at no extra cost
* A full refund, including your original shipping costs

Let us know which option you prefer, and we will process it immediately.

The Support Team


## Task 3: Data Analysis with Structured (JSON) Output

### Iteration 1 — Baseline (free text)

In [8]:
sales_table = """
Region,Q1_Sales,Q2_Sales
North,120000,135000
South,98000,91000
East,150000,162000
West,87000,79000
"""

response = client.models.generate_content(
    model=MODEL,
    contents="Analyze this sales data: " + sales_table,
    config=types.GenerateContentConfig(
        thinking_config=THINKING,
        temperature=1.0,
        top_p=0.95,
    ),
)
print(response.text)


Here is an analysis of the provided sales data by region for Q1 and Q2.

### **1. Summary Table (with Total Sales and Growth)**

| Region | Q1 Sales ($) | Q2 Sales ($) | Total Sales ($) | Q-o-Q Growth ($) | Q-o-Q Growth (%) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **East** | 150,000 | 162,000 | 312,000 | +12,000 | +8.0% |
| **North** | 120,000 | 135,000 | 255,000 | +15,000 | +12.5% |
| **South** | 98,000 | 91,000 | 189,000 | -7,000 | -7.1% |
| **West** | 87,000 | 79,000 | 166,000 | -8,000 | -9.2% |
| **Total** | **455,000** | **467,000** | **922,000** | **+12,000** | **+2.6%** |

---

### **2. Key Takeaways & Insights**

* **Overall Growth:** Total sales across all regions increased by **2.6%** from Q1 ($455,000) to Q2 ($467,000), driven entirely by strong performances in the North and East regions.
* **Top Performing Region (East):** The East region generated the highest total revenue ($312,000) and maintained the #1 spot in both quarters, growing by **8.0%** in Q2.
* **Fastest

### Iteration 2 — Structured JSON output with a schema

In [9]:
schema = {
    "type": "object",
    "properties": {
        "regions": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "region": {"type": "string"},
                    "q1_sales": {"type": "number"},
                    "q2_sales": {"type": "number"},
                    "pct_change": {"type": "number"},
                },
                "required": ["region", "q1_sales", "q2_sales", "pct_change"],
            },
        },
        "best_performing_region": {"type": "string"},
        "worst_performing_region": {"type": "string"},
    },
    "required": ["regions", "best_performing_region", "worst_performing_region"],
}

prompt = (
    "Analyze the sales data below. pct_change = (Q2 - Q1) / Q1 * 100, rounded to 1 decimal.\n\n"
    + sales_table
)

response = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config=types.GenerateContentConfig(
        thinking_config=THINKING,
        temperature=0.1,
        response_mime_type="application/json",
        response_schema=schema,
    ),
)

parsed = json.loads(response.text)
print(json.dumps(parsed, indent=2))


{
  "regions": [
    {
      "region": "North",
      "q1_sales": 120000,
      "q2_sales": 135000,
      "pct_change": 12.5
    },
    {
      "region": "South",
      "q1_sales": 98000,
      "q2_sales": 91000,
      "pct_change": -7.1
    },
    {
      "region": "East",
      "q1_sales": 150000,
      "q2_sales": 162000,
      "pct_change": 8.0
    },
    {
      "region": "West",
      "q1_sales": 87000,
      "q2_sales": 79000,
      "pct_change": -9.2
    }
  ],
  "best_performing_region": "North",
  "worst_performing_region": "West"
}
